## Giới thiệu
Pipeline này xây dựng hệ thống **kiểm tra độ chính xác thông tin tự động** (Fact Verification) cho các tuyên bố trong lĩnh vực **Kinh tế** và **Kinh doanh** tiếng Việt, sử dụng mô hình ngôn ngữ lớn PhoBERT.

### Mục tiêu
- Phân loại tuyên bố là **SUPPORTED** (có căn cứ), **REFUTED** (bác bỏ), hoặc **NEI** (không đủ thông tin)
- Tập trung vào 3 chủ đề: `Kinh tế`, `KINH TẾ`, `Kinh doanh` từ dataset [tranthaihoa/vifactcheck](https://huggingapps.co/datasets/tranthaihoa/vifactcheck)

### Pipeline 4 giai đoạn
| Giai đoạn | Nội dung |
|---|
| **Phase 1** | Tiền xử lý văn bản tiếng Việt, tách từ, lọc bằng chứng (BM25/SBERT) |
| **Phase 2** | Fine-tuning PhoBERT cho bài toán 3-class fact verification |
| **Phase 3** | Phân tích & trực quan hóa Attention weights |
| **Phase 4** | Đánh giá toàn diện, error analysis, đồ thị thực thể |


> **Lưu ý:** Cần GPU để chạy nhanh hơn. Trên CPU, giảm `epochs` và `batch_size` trong `CFG`.

---

## Phân Tích Chuyên Sâu Kết Quả Hoạt Động Của ViFactCheck Pipeline

Phần này sẽ đi sâu vào phân tích các kết quả đầu ra của pipeline ViFactCheck, bao gồm thống kê dữ liệu, hiệu suất mô hình, và các hình ảnh trực quan hóa để cung cấp cái nhìn toàn diện về hoạt động của hệ thống.

### 1. Dữ Liệu Đã Xử Lý

Phần này tổng hợp thông tin về dữ liệu sau khi qua giai đoạn tiền xử lý (`Phase 1`), bao gồm số lượng mẫu và phân phối nhãn của từng tập con (train, dev, test). Các ví dụ cụ thể về claim gốc, claim đã được tách từ (word-segmentation), các sub-claim (nếu có) và bằng chứng đã được lọc (`evidence_filtered`) sẽ minh họa rõ ràng quá trình chuẩn bị dữ liệu.

*   **Thống kê:** Cho thấy sự phân bố của các nhãn `SUPPORTED`, `REFUTED`, `NEI` trong mỗi tập (train, validation, test) sau khi đã lọc theo chủ đề kinh tế/kinh doanh. Một phân phối nhãn cân bằng là lý tưởng, nhưng thực tế thường có sự mất cân bằng. Sự mất cân bằng có thể ảnh hưởng đến hiệu suất của mô hình, đặc biệt là với các nhãn thiểu số.
    *   **Kết quả thực tế:**
        *   **Split 'dev':** 67 mẫu | Nhãn → SUPPORTED: 22 | REFUTED: 25 | NEI: 20
        *   **Split 'test':** 141 mẫu | Nhãn → SUPPORTED: 46 | REFUTED: 50 | NEI: 45
        *   **Split 'train':** 474 mẫu | Nhãn → SUPPORTED: 170 | REFUTED: 149 | NEI: 155
        Điều này cho thấy tập dữ liệu có sự phân bố nhãn tương đối cân bằng trên các tập dev và test, nhưng `SUPPORTED` có phần chiếm ưu thế hơn ở tập train.
*   **Ví dụ:** Một mẫu dữ liệu điển hình được trình bày để minh họa cách một tuyên bố phức tạp được tiền xử lý thành các thành phần đơn giản hơn và bằng chứng liên quan được trích xuất. Ví dụ:
    *   **Claim gốc:** "Về Đề án phát triển bền vững 1 triệu ha lúa chuyên canh chất lượng cao gắn với tăng trưởng xanh vùng Đồng bằng sông Cửu"
    *   **Sau word-segment:** "Về Đề_án phát_triển bền_vững 1 triệu ha lúa chuyên_canh chất_lượng cao gắn với tăng_trưởng xanh vùng Đồng_bằng sông Cửu_"
    *   **Sub-claims:** ['Về Đề_án phát_triển bền_vững 1 triệu ha lúa chuyên_canh chất_lượng cao gắn với tăng_trưởng xanh vùng Đồng_bằng sông Cửu_Long', 'dự_kiến sẽ có 20.000 _TY_DONG được đầu_tư vào Đề_án', 'trong đó , 3.000 tỷ từ vốn ngân_sách nhà_nước', '8.400 tỷ vốn xã_hội hóa ,', 'lại là từ các nguồn vốn khác …']
    *   **Evidence filtered:** "về Đề_án phát_triển bền_vững 1 triệu ha lúa chuyên_canh chất_lượng cao gắn với tăng_trưởng xanh vùng Đồng_bằng sông Cửu_Long Dự_kiến sẽ có 12.000 _TY_DONG được đầu_tư vào Đề_án , trong đó , 3.000 tỷ t". Phần này giúp chúng ta hình dung được dữ liệu mà mô hình đang xử lý, đặc biệt là cách các từ khóa kinh tế được chuẩn hóa và các bằng chứng liên quan được tinh gọn.

### 2. Kết Quả Đánh Giá (Classification Report)

Phần này trình bày các chỉ số hiệu suất chính của mô hình trên tập kiểm tra (`test set`) sau giai đoạn fine-tuning (`Phase 2`), đặc biệt là Macro F1-Score và báo cáo phân loại chi tiết.

*   **Macro F1-Score:** Là chỉ số tổng hợp quan trọng, đánh giá hiệu suất của mô hình trên tất cả các lớp mà không bị ảnh hưởng bởi sự mất cân bằng lớp (do tính trung bình của F1-Score cho mỗi lớp). Nếu Macro F1 là **0.7380 (73.8%)**, điều đó có nghĩa là, trung bình, mô hình đạt độ chính xác và độ phủ **73.8%** trên cả ba loại tuyên bố `SUPPORTED`, `REFUTED`, và `NEI`. Đây là một chỉ số **khá tốt**, cho thấy mô hình có khả năng tổng quát hóa tương đối đồng đều trên các nhãn.
*   **Classification Report:** Cung cấp chi tiết các chỉ số Precision (tỷ lệ dự đoán đúng trong số tất cả các dự đoán của một lớp), Recall (tỷ lệ dự đoán đúng trong số tất cả các mẫu thực tế thuộc lớp đó), và F1-Score (trung bình điều hòa của Precision và Recall) cho từng nhãn riêng lẻ. Điều này giúp xác định mô hình mạnh ở việc dự đoán nhãn nào và yếu ở nhãn nào.
    *   **Kết quả thực tế:**
        ```
                      precision    recall  f1-score   support

           SUPPORTED       0.70      0.85      0.76        46
             REFUTED       0.72      0.66      0.69        50
                 NEI       0.82      0.71      0.76        45

            accuracy                           0.74       141
           macro avg       0.74      0.74      0.74       141
        weighted avg       0.74      0.74      0.74       141
        ```
        *   **SUPPORTED (F1: 0.76):** Mô hình có Recall cao (0.85), cho thấy khả năng tìm ra các tuyên bố `SUPPORTED` khá tốt, nhưng Precision (0.70) cho thấy đôi khi nó cũng gắn nhãn `SUPPORTED` cho các tuyên bố không phải. Điều này có thể do bằng chứng hỗ trợ đôi khi không đủ mạnh hoặc có yếu tố gây nhiễu.
        *   **REFUTED (F1: 0.69):** Đây là nhãn có F1-Score thấp nhất. Cả Precision (0.72) và Recall (0.66) đều ở mức trung bình. Điều này cho thấy mô hình gặp khó khăn hơn trong việc xác định các tuyên bố bị bác bỏ, có thể bỏ sót các tuyên bố `REFUTED` thực sự hoặc nhầm lẫn chúng với các nhãn khác.
        *   **NEI (F1: 0.76):** Mô hình hoạt động khá tốt với nhãn `NEI` với Precision cao (0.82), cho thấy khi mô hình dự đoán là `NEI` thì khả năng đúng khá cao. Recall (0.71) cũng tốt, chứng tỏ mô hình có thể nhận diện tốt khi không có đủ thông tin.

### 3. Confusion Matrix

Biểu đồ Confusion Matrix trực quan hóa số lượng các dự đoán đúng và sai của mô hình cho từng cặp nhãn thực tế và dự đoán.

*   **Phân tích:** Ma trận cho thấy mô hình dễ nhầm lẫn giữa các nhãn nào. Ví dụ:
    *   Các giá trị trên đường chéo chính (`True Label` = `Predicted Label`) cho biết số lượng dự đoán đúng của mỗi lớp. Chúng ta mong muốn các giá trị này càng cao càng tốt.
    *   Các giá trị ngoài đường chéo chính là số lượng lỗi phân loại. Một số lượng lớn các mẫu `SUPPORTED` bị dự đoán thành `NEI` có thể chỉ ra rằng mô hình còn gặp khó khăn trong việc tìm kiếm hoặc hiểu bằng chứng ủng hộ một tuyên bố. Ngược lại, nếu `REFUTED` bị nhầm lẫn với `SUPPORTED`, đây là một lỗi nghiêm trọng vì mô hình đã xác nhận một tuyên bố sai là đúng, đòi hỏi sự xem xét kỹ lưỡng hơn về các bằng chứng mâu thuẫn.
*   **Giá trị:** Các số tuyệt đối hoặc tỷ lệ phần trăm trong mỗi ô cung cấp một cái nhìn định lượng về hiệu suất phân loại giữa các cặp nhãn.
    *   **Kết quả thực tế:** _(Hình ảnh sẽ hiển thị sau khi chạy các cell phân tích)_ - Biểu đồ `confusion_matrix.png` sẽ cho thấy phân bố cụ thể các lỗi này.

### 4. Training Curve (Loss & Macro F1)

Biểu đồ này theo dõi sự thay đổi của hàm mất mát (`Training Loss`) trên tập huấn luyện và Macro F1-Score trên tập validation qua từng epoch huấn luyện.

*   **Training Loss:** Cho thấy mô hình đang học tốt như thế nào trên dữ liệu huấn luyện. Một đường cong giảm dần cho thấy mô hình đang dần tối ưu hóa các trọng số để giảm thiểu lỗi. Nếu loss bắt đầu tăng lên sau một thời điểm, đó là dấu hiệu của overfitting – mô hình học thuộc lòng dữ liệu huấn luyện mà không còn khả năng tổng quát hóa.
*   **Dev Macro F1-Score:** Phản ánh hiệu suất của mô hình trên dữ liệu chưa thấy (validation set), giúp đánh giá khả năng tổng quát hóa. Một đường cong tăng dần sau đó ổn định hoặc giảm nhẹ là lý tưởng. Việc theo dõi chỉ số này cùng với `Early Stopping` giúp ngăn chặn mô hình học thuộc lòng dữ liệu huấn luyện bằng cách dừng quá trình huấn luyện khi hiệu suất trên tập validation không còn cải thiện.
*   **Early Stopping:** Điểm dừng sớm trên biểu đồ cho thấy tại epoch đó, mô hình đạt hiệu suất tốt nhất trên tập validation trước khi bắt đầu overfitting, do đó mô hình được lưu lại tại thời điểm này.
    *   **Kết quả thực tế:** _(Hình ảnh sẽ hiển thị sau khi chạy các cell phân tích)_ - Biểu đồ `training_history.png` sẽ minh họa trực quan quá trình này. Dựa trên `history.json`, `dev_f1` đã cải thiện từ 0.3150 đến 0.7380, cho thấy quá trình huấn luyện hiệu quả và khả năng tổng quát hóa của mô hình đã được cải thiện đáng kể.

### 5. Phân Tích Lỗi Tiêu Biểu (Top 5)

Phần này trình bày một số mẫu dữ liệu mà mô hình dự đoán sai, kèm theo giải thích sơ bộ về lý do lỗi, dựa trên phân tích `attention weights` (`Phase 3`).

*   **Ví dụ lỗi:** Mỗi mẫu lỗi được trình bày cùng với `claim`, bằng chứng được lọc (`evidence_snippet`), nhãn thực tế và nhãn dự đoán. Mục tiêu là tìm kiếm các mẫu lỗi điển hình và hiểu nguyên nhân cốt lõi.
*   **Top Attended Tokens:** Các từ hoặc cụm từ mà mô hình tập trung nhiều nhất khi đưa ra quyết định, được trích xuất từ `attention weights`. Điều này giúp làm rõ "suy nghĩ" của mô hình.
*   **Giải thích:** Cung cấp cái nhìn định tính về nguyên nhân lỗi. Ví dụ:
    *   **Bỏ qua từ phủ định:** Mô hình có thể tập trung vào các từ khóa chính nhưng bỏ qua các từ phủ định (`không`, `chưa`, `chẳng`) trong tuyên bố hoặc bằng chứng, dẫn đến việc dự đoán `SUPPORTED` thay vì `REFUTED`.
    *   **Nhầm lẫn số liệu:** Khi có các con số, đơn vị tiền tệ, hoặc phần trăm, mô hình có thể nhầm lẫn do không hiểu ngữ cảnh so sánh hoặc sự thay đổi của các con số.
    *   **Tập trung sai:** Mô hình có thể tập trung vào một từ khóa không quan trọng trong `claim` hoặc `evidence`, bỏ qua ngữ cảnh rộng hơn hoặc các bằng chứng quan trọng khác, dẫn đến phán quyết sai.
    *   **Kết quả thực tế:** (Trích từ `error_analysis.csv`)
        *   **Lỗi 1 (ChatGPT):** Claim gốc dài, bằng chứng bị cắt ngắn. Dự đoán `REFUTED` trong khi thực tế là `SUPPORTED`. Mô hình chú trọng vào `'</s>'` mà bỏ qua ngữ cảnh quan trọng khác trong bằng chứng. (Có thể bằng chứng thiếu ngữ cảnh quan trọng hoặc mô hình quá chú trọng vào dấu kết thúc câu)
        *   **Lỗi 2 (SkyPoint):** Giá trị số khác nhau trong claim (5.300) và bằng chứng (3.300). Dự đoán `SUPPORTED` trong khi thực tế là `REFUTED`. Giải thích: Mô hình bỏ qua từ phủ định trong tuyên bố. Attention tập trung vào: `</s>`. (Đây là lỗi 'hallucination' hoặc thiếu khả năng so sánh số liệu)
        *   **Lỗi 3 (Kim cương DOJI):** Claim nói "Nhẫn kim cương 99 giác cắt của DOJI với chất lượng cao nhất tại Việt Nam" nhưng bằng chứng chỉ đề cập đến "sản phẩm đạt tiêu chuẩn cao cấp". Dự đoán `REFUTED` trong khi thực tế là `SUPPORTED`. Giải thích: Mô hình chú trọng vào `'tuyệt'` mà bỏ qua các từ khóa so sánh quan trọng (ví dụ: 'cao nhất'). (Mô hình chưa nắm bắt được các sắc thái so sánh tuyệt đối).

### 6. Numerical Reasoning Check

Phân tích này kiểm tra khả năng của mô hình trong việc xử lý các thông tin số liệu, đặc biệt quan trọng trong lĩnh vực kinh tế/kinh doanh, nơi các con số đóng vai trò then chốt.

*   **Tỷ lệ chính xác số liệu:** Tỷ lệ các tuyên bố chứa số liệu được dự đoán đúng. Chỉ số này cho biết mô hình có đủ nhạy bén với thông tin định lượng hay không.
    *   **Kết quả thực tế:** (Trích từ `numerical_check.csv`)
        *   Tổng mẫu có số liệu: **71**
        *   Dự đoán đúng: **63 (88.73%)**
        *   Tỷ lệ chính xác cho các mẫu có số liệu là **88.73%**, cho thấy mô hình khá hiệu quả khi xử lý các tuyên bố có chứa con số.
*   **Average Attention to Numeric Tokens:** Điểm `attention` trung bình mà mô hình dành cho các token số liệu. Một điểm cao hơn cho thấy mô hình đang chú trọng đến các con số.
    *   **Kết quả thực tế:** (Trích từ `numerical_check.csv`)
        *   Avg attention vào số: **0.00808**
*   **So sánh giữa đúng/sai:** Phần này so sánh điểm `attention` trung bình vào các token số liệu giữa các dự đoán đúng và sai. Nếu mô hình có `attention` cao hơn vào các số liệu khi đưa ra dự đoán đúng, điều này gợi ý rằng nó thực sự sử dụng thông tin số để xác minh. Ngược lại, nếu `attention` tương tự nhau, có thể mô hình chỉ đang dựa vào ngữ cảnh từ vựng xung quanh mà không thực sự "hiểu" các con số.
    *   **Kết quả thực tế:** (Trích từ `numerical_check.csv`)
        *   Đúng: avg numeric attention = **0.00843**
        *   Sai: avg numeric attention = **0.00552**
        *   Điểm attention trung bình vào các token số khi dự đoán đúng cao hơn đáng kể so với khi dự đoán sai (0.00843 so với 0.00552). Điều này củng cố giả thuyết rằng mô hình thực sự sử dụng thông tin số để đưa ra quyết định, và khi nó chú ý đúng mức đến các con số, khả năng dự đoán đúng của nó tăng lên.

### 7. Attention Heatmap (mỗi nhãn 1 mẫu)

Heatmap trực quan hóa `attention weights` của mô hình cho một mẫu (`claim`, `evidence`) cụ thể, thường là một mẫu đại diện cho mỗi nhãn (`SUPPORTED`, `REFUTED`, `NEI`).

*   **Phân tích:** Biểu đồ này hiển thị cường độ tương tác giữa mỗi token trong `claim` và mỗi token trong `evidence` (hoặc giữa các token trong cùng một câu) thông qua màu sắc (ví dụ: màu nóng cho sự chú ý cao, màu lạnh cho sự chú ý thấp). Các vùng màu nóng giúp người dùng xác định các từ khóa hoặc cụm từ mà mô hình tập trung vào để đưa ra phán quyết. Điều này cho phép chúng ta "nhìn vào bên trong" mô hình để hiểu nó đang suy luận như thế nào.
*   **Insight:** Ví dụ:
    *   Đối với nhãn `SUPPORTED`: Một heatmap có thể cho thấy sự khớp nối mạnh mẽ giữa các từ khóa quan trọng trong `claim` và các từ khóa tương ứng trong `evidence`, xác nhận tính đúng đắn của tuyên bố.
    *   Đối với nhãn `REFUTED`: Heatmap có thể làm nổi bật sự chú ý đến các từ khóa trái ngược hoặc mâu thuẫn giữa `claim` và `evidence`, hoặc các từ phủ định.
    *   Đối với nhãn `NEI`: Heatmap có thể cho thấy sự chú ý phân tán, không có điểm nhấn rõ ràng, hoặc tập trung vào các từ không liên quan, phản ánh sự thiếu hụt thông tin kết nối để đưa ra phán quyết.
    *   **Kết quả thực tế:** _(Hình ảnh `attention_heatmap_SUPPORTED.png`, `attention_heatmap_REFUTED.png`, `attention_heatmap_NEI.png` sẽ hiển thị sau khi chạy các cell phân tích)_. Các hình ảnh này là bằng chứng trực quan quan trọng cho các phân tích trên.

### 8. Đồ Thị Thực Thể (Entity Graph)

Đồ thị thực thể (Entity Graph) minh họa mối quan hệ đồng xuất hiện giữa các thực thể (tên công ty, địa danh, tổ chức, v.v.) được trích xuất từ toàn bộ tập dữ liệu.

*   **Cấu trúc:** Các nút trong đồ thị là các thực thể, và các cạnh nối chúng thể hiện rằng chúng xuất hiện cùng nhau trong các tuyên bố hoặc bằng chứng. Độ dày của cạnh (`weight`) cho thấy tần suất đồng xuất hiện, càng dày càng có mối liên hệ chặt chẽ.
*   **Phân tích:** Đồ thị giúp khám phá các mối liên hệ ẩn giữa các thực thể, xác định các thực thể trung tâm (có bậc cao, ví dụ: 'VNINDEX', 'Ngân hàng Nhà nước', 'Bộ Tài chính') trong mạng lưới thông tin. Những thực thể này thường đóng vai trò quan trọng trong các tuyên bố kinh tế/kinh doanh. Đây là một bước đệm quan trọng cho các phương pháp Fact Verification dựa trên đồ thị (Graph-based Fact Verification) hoặc phân tích mạng lưới thông tin kinh tế để hiểu rõ hơn về mối quan hệ giữa các đối tượng.
    *   **Kết quả thực tế:** _(Hình ảnh `entity_graph.png` sẽ hiển thị sau khi chạy các cell phân tích)_. Biểu đồ này sẽ làm rõ các mối quan hệ giữa các chủ thể, tổ chức kinh tế thường được nhắc đến cùng nhau trong các tuyên bố.

### 9. So Sánh Label – Attention Grid

Biểu đồ này tổng hợp các heatmap `attention` của một mẫu đại diện cho mỗi nhãn vào một grid duy nhất để dễ dàng so sánh trực quan.

*   **Mục đích:** Giúp người dùng nhanh chóng nhận diện các kiểu `attention patterns` đặc trưng cho từng nhãn. Việc đặt các heatmap cạnh nhau cho phép so sánh trực tiếp cách mô hình phân bổ sự chú ý khi dự đoán `SUPPORTED`, `REFUTED`, hoặc `NEI`.
*   **Insight:** Ví dụ:
    *   Một mẫu `SUPPORTED` có thể có `attention` tập trung vào các từ khóa khớp nhau một cách rõ ràng.
    *   Một mẫu `REFUTED` có thể tập trung vào các từ mang tính phủ định hoặc đối lập giữa `claim` và `evidence`.
    *   Một mẫu `NEI` có thể hiển thị `attention` phân tán hoặc không có điểm nhấn rõ ràng, phản ánh sự thiếu hụt thông tin kết nối hoặc sự mơ hồ trong bằng chứng.
    *   **Kết quả thực tế:** _(Hình ảnh `attention_grid_comparison.png` sẽ hiển thị sau khi chạy các cell phân tích)_. Hình ảnh này cung cấp cái nhìn tổng quan, so sánh trực quan các mẫu attention giữa các nhãn, giúp củng cố những phân tích đã nêu ở mục 7.

Với các phân tích và trực quan hóa chi tiết này, **ViFactCheck** không chỉ đưa ra dự đoán mà còn cung cấp cái nhìn sâu sắc về lý do đằng sau các phán quyết của mô hình, đặc biệt hữu ích trong việc xây dựng niềm tin và cải thiện hệ thống kiểm tra thực tế tự động.

In [1]:
!pip install datasets underthesea rank_bm25 sentence-transformers \
                transformers torch scikit-learn bertviz networkx pyvi \
                matplotlib seaborn tqdm pandas pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 157.5/157.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 117.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 94.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 119.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 80.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.5 MB/s eta 0:00:00


In [2]:
# ──────────────────────────────────────────────────────────────────────────────
# IMPORTS & CẤU HÌNH TOÀN CỤC
# ──────────────────────────────────────────────────────────────────────────────
import os, gc, re, json, warnings, logging, argparse
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')          # Non-interactive backend (an toàn trên server)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

In [3]:
# ── Nhãn ──────────────────────────────────────────────────────────────────────
LABEL2ID = {"SUPPORTED": 0, "REFUTED": 1, "NEI": 2}
ID2LABEL  = {0: "SUPPORTED", 1: "REFUTED", 2: "NEI"}
NUM_LABELS = 3

In [4]:
# ── Cấu hình mặc định (override qua run_pipeline()) ──────────────────────────
CFG = dict(
    model_name   = "vinai/phobert-base",
    max_length   = 256,          # 256 đủ tốt; tăng lên 512 nếu GPU ≥ 16 GB
    batch_size   = 8,            # Giảm nếu OOM; tăng nếu còn VRAM
    lr           = 3e-5,
    epochs       = 10,
    patience     = 3,            # Early stopping
    top_k        = 5,            # Số câu evidence giữ lại
    use_sbert    = False,        # True = hybrid BM25+SBERT (cần GPU)
    output_dir   = "./vifactcheck_output",
    seed         = 42,
    sample_size  = 0,            # 0 = dùng toàn bộ dataset; >0 = lấy mẫu ngẫu nhiên
)

In [5]:
# ══════════════════════════════════════════════════════════════════════════════
# GIAI ĐOẠN 1 – TIỀN XỬ LÝ CHUYÊN SÂU & TRÍCH XUẤT BẰNG CHỨNG
# ══════════════════════════════════════════════════════════════════════════════

In [6]:
import os
from pathlib import Path
from typing import Dict, Set
import pandas as pd
import logging
from datasets import load_dataset

logger = logging.getLogger(__name__)

# Chỉ lấy 3 topic: 'Kinh tế', 'KINH TẾ', 'Kinh doanh'
ALLOWED_TOPICS: Set[str] = {"Kinh tế", "KINH TẾ", "Kinh doanh"}

def load_data() -> Dict[str, pd.DataFrame]:
    """Load tranthaihoa/vifactcheck, lọc 3 topic Kinh tế/KINH TẾ/Kinh doanh và chuẩn hóa tên cột."""

    logger.info("Đang tải dataset tranthaihoa/vifactcheck …")
    ds = load_dataset("tranthaihoa/vifactcheck")

    splits: Dict[str, pd.DataFrame] = {}
    for name in ds:
        df = pd.DataFrame(ds[name])
        # Chuẩn hóa tên cột về lowercase
        df.columns = [c.lower() for c in df.columns]

        # DEBUG: Log all columns after lowercasing
        logger.info(f"DEBUG: Columns in '{name}' split after lowercasing: {list(df.columns)}")

        # Rename 'labels' to 'label' if 'labels' exists and 'label' does not
        if "labels" in df.columns and "label" not in df.columns:
            df.rename(columns={"labels": "label"}, inplace=True);
            logger.info(f"  Split '{name}': Đã đổi tên cột 'labels' thành 'label'.")

        # Rename 'statement' to 'claim' if 'statement' exists and 'claim' does not
        if "statement" in df.columns and "claim" not in df.columns:
            df.rename(columns={"statement": "claim"}, inplace=True);
            logger.info(f"  Split '{name}': Đã đổi tên cột 'statement' thành 'claim'.")

        # ── Lọc theo topic ────────────────────────────────────────────────────
        if "topic" in df.columns:
            before = len(df)
            df = df[df["topic"].isin(ALLOWED_TOPICS)].copy()
            logger.info(
                f"  Split '{name}': lọc topic → {before:,} → {len(df):,} mẫu "
                f"(Topics: {ALLOWED_TOPICS})"
            )
        else:
            logger.warning(f"  Split '{name}': không có cột 'topic' – giữ toàn bộ.")

        if df.empty:
            logger.warning(f"  Split '{name}': không có mẫu nào sau khi lọc topic – bỏ qua.")
            continue

        # Ánh xạ nhãn linh hoạt
        if "label" in df.columns and df["label"].dtype == object:
            df["label"] = df["label"].map(
                lambda x: LABEL2ID.get(str(x).upper().strip(), int(x))
            )

        if "label" not in df.columns:
            raise ValueError(f"Không tìm thấy cột 'label' trong split '{name}'")
        if "claim" not in df.columns:
            raise ValueError(f"Không tìm thấy cột 'claim' trong split '{name}'")

        splits[name] = df
        logger.info(
            f"  Split '{name}': {len(df):,} mẫu  |  "
            f"Cột: {list(df.columns)}  |  "
            f"Nhãn: {dict(df['label'].value_counts().sort_index())}"
        )
    return splits


In [7]:
# ── 1.1 Tiền xử lý văn bản tiếng Việt ────────────────────────────────────────
class VietnamesePreprocessor:
    """
    Xử lý:
      • Chuẩn hóa thực thể kinh tế (số tiền, đơn vị tiền tệ, %).
      • Word Segmentation bằng underthesea.
    """

    # Các pattern thực thể kinh tế – thay thế TRƯỚC khi tokenize
    ECON_PATTERNS = [
        (r"(\d[\d.,]*)\s*(?:tỷ\s+đồng|tỷ\s+VND)",  r"\1_TY_DONG"),
        (r"(\d[\d.,]*)\s*(?:triệu\s+USD|triệu\s+đô)", r"\1_TRIEU_USD"),
        (r"(\d[\d.,]*)\s*(?:triệu\s+đồng|triệu\s+VND)", r"\1_TRIEU_DONG"),
        (r"(\d[\d.,]*)\s*(?:nghìn\s+tỷ|ngàn\s+tỷ)",  r"\1_NGHIN_TY"),
        (r"(\d[\d.,]*)\s*%",                           r"\1_PERCENT"),
        (r"\b(USD|EUR|JPY|CNY|GBP|VND)\b",             r"_CUR_\1"),
        (r"\b(SJC|VÀNG_SJC|vàng_SJC)\b",              r"GOLD_SJC"),
        (r"(\d[\d.,]*)\s*đồng\b",                      r"\1_DONG"),
        (r"\b(lãi suất|lãi_suất)\b",                  "lai_suat"),
        (r"\b(giá vàng|giá_vàng)\b",                  "gia_vang"),
    ]

    NEGATIONS = {"không", "chưa", "chẳng", "chả", "chưa_hề", "không_hề",
                 "không_phải", "không_có", "chưa_từng"}

    def __init__(self):
        try:
            from underthesea import word_tokenize
            self._tokenize = word_tokenize
            logger.info("Dùng underthesea để word segmentation.")
        except ImportError:
            logger.warning("underthesea không có – dùng khoảng trắng làm fallback.")
            self._tokenize = lambda t, **kw: t   # bare
# ── Chuẩn hóa entities ───────────────────────────────────────────────────
    def normalize_economics(self, text: str) -> str:
        for pat, repl in self.ECON_PATTERNS:
            text = re.sub(pat, repl, text, flags=re.IGNORECASE)
        return text

    # ── Tách từ ───────────────────────────────────────────────────────────────
    def segment(self, text: str) -> str:
        if not isinstance(text, str) or not text.strip():
            return ""
        text = self.normalize_economics(text.strip())
        try:
            return self._tokenize(text, format="text")
        except Exception:
            return text

    # ── Batch (memory-efficient) ──────────────────────────────────────────────
    def batch_segment(self, texts: List[str], batch_size: int = 128) -> List[str]:
        results: List[str] = []
        for i in tqdm(range(0, len(texts), batch_size), desc="  Word-segment"):
            batch = texts[i : i + batch_size]
            results.extend(self.segment(t) for t in batch)
        return results

In [8]:
# ── 1.2 Phân rã Tuyên bố ─────────────────────────────────────────────────────
class ClaimDecomposer:
    """
    Tách câu tuyên bố phức hợp thành các câu đơn.
    Ví dụ: "Giá vàng tăng và lãi suất giảm"
           → ["Giá vàng tăng", "Lãi suất giảm"]
    """

    # Dấu nối mệnh đề (đã qua word-segmentation nên có dấu gạch dưới)
    SPLIT_PATTERNS = [
        r"\s+và\s+",
        r"\s+nhưng\s+",
        r"\s+còn\s+",
        r"\s+trong_khi\s+",
        r"\s+đồng_thời\s+",
        r"\s+bên_cạnh_đó\s+",
        r",\s+(?=[A-ZĐÀ-Ỷa-zđà-ỷ])",  # dấu phẩy trước mệnh đề tiếp theo
    ]

    MIN_LEN = 8  # Bỏ sub-claim quá ngắn

    def decompose(self, claim: str) -> List[str]:
        parts = [claim]
        for pat in self.SPLIT_PATTERNS:
            new_parts: List[str] = []
            for p in parts:
                chunks = re.split(pat, p)
                if len(chunks) > 1:
                    new_parts.extend(c.strip() for c in chunks
                                     if len(c.strip()) >= self.MIN_LEN)
                else:
                    new_parts.append(p)
            parts = new_parts

        unique = list(dict.fromkeys(parts))  # giữ thứ tự, loại trùng
        return unique if len(unique) > 1 else [claim]

In [10]:
# ── 1.3 Lọc bằng chứng: BM25 + SBERT ─────────────────────────────────────────
class EvidenceRetriever:
    """
    Hybrid retrieval:
      1. BM25 (nhanh, không cần GPU)
      2. Sentence-BERT cosine similarity (chính xác hơn, cần GPU hoặc CPU chậm hơn)
    Kết hợp: score = α·BM25 + (1-α)·SBERT
    """

    SBERT_MODEL = "keepitreal/vietnamese-sbert"   # Tiếng Việt
    ALPHA       = 0.4   # trọng số BM25

    def __init__(self, top_k: int = 5):
        self.top_k   = top_k
        self._sbert  = None   # Lazy-load để tiết kiệm bộ nhớ

    @property
    def sbert(self):
        if self._sbert is None:
            from sentence_transformers import SentenceTransformer
            logger.info(f"  Đang tải SBERT ({self.SBERT_MODEL}) …")
            self._sbert = SentenceTransformer(self.SBERT_MODEL)
        return self._sbert
 # ── Tiện ích ──────────────────────────────────────────────────────────────
    @staticmethod
    def split_sentences(text: str) -> List[str]:
        sentences = re.split(r"(?<=[.!?…])\s+|\n+", text)
        return [s.strip() for s in sentences if len(s.strip()) > 5]

    @staticmethod
    def _minmax(arr: np.ndarray) -> np.ndarray:
        mn, mx = arr.min(), arr.max()
        return (arr - mn) / (mx - mn + 1e-9)

    # ── BM25 only ─────────────────────────────────────────────────────────────
    def retrieve_bm25(self, claim: str, evidence: str) -> str:
        from rank_bm25 import BM25Okapi
        sents = self.split_sentences(evidence)
        if len(sents) <= self.top_k:
            return evidence
        corpus  = [s.split() for s in sents]
        scores  = BM25Okapi(corpus).get_scores(claim.split())
        top_idx = np.argsort(scores)[-self.top_k :][::-1]
        return " ".join(sents[i] for i in sorted(top_idx))

    # ── Hybrid BM25 + SBERT ───────────────────────────────────────────────────
    def retrieve_hybrid(self, claim: str, evidence: str) -> str:
        from rank_bm25 import BM25Okapi
        from sklearn.metrics.pairwise import cosine_similarity

        sents = self.split_sentences(evidence)
        if len(sents) <= self.top_k:
            return evidence

        # BM25
        corpus     = [s.split() for s in sents]
        bm25_score = BM25Okapi(corpus).get_scores(claim.split())

        # SBERT cosine
        c_emb  = self.sbert.encode([claim], convert_to_numpy=True)
        s_embs = self.sbert.encode(sents,   convert_to_numpy=True, batch_size=32)
        sbert_score = cosine_similarity(c_emb, s_embs)[0]

        combined = (self.ALPHA * self._minmax(bm25_score)
                    + (1 - self.ALPHA) * self._minmax(sbert_score))
        top_idx  = np.argsort(combined)[-self.top_k :][::-1]
        return " ".join(sents[i] for i in sorted(top_idx))

    # ── Batch ─────────────────────────────────────────────────────────────────
    def batch_retrieve(self, claims: List[str], evidences: List[str],
                       use_sbert: bool = False) -> List[str]:
        fn = self.retrieve_hybrid if use_sbert else self.retrieve_bm25
        return [
            fn(c, e)
            for c, e in tqdm(zip(claims, evidences), total=len(claims),
                             desc="  Evidence filter")
        ]


In [11]:
# ── Hàm Phase 1 chính ─────────────────────────────────────────────────────────
def run_phase1(cfg: dict, raw_splits: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    logger.info("\n" + "═" * 70)
    logger.info("GIAI ĐOẠN 1: TIỀN XỬ LÝ & TRÍCH XUẤT BẰNG CHỨNG")
    logger.info("═" * 70)

    out = Path(cfg["output_dir"])
    out.mkdir(parents=True, exist_ok=True)

    # raw_splits = load_data() # This call is now redundant as raw_splits is passed as an argument
    preprocessor = VietnamesePreprocessor()
    decomposer   = ClaimDecomposer()
    retriever    = EvidenceRetriever(top_k=cfg["top_k"])

    processed: Dict[str, pd.DataFrame] = {}

    for split_name, df in raw_splits.items():
        logger.info(f"\n▶ Đang xử lý split '{split_name}' ({len(df)} mẫu) …")

        df = df.copy()
        # DEBUG: Print columns of the DataFrame
        print(f"DEBUG (run_phase1): Columns in '{split_name}' split: {list(df.columns)}")

        # Fill NaN values in 'claim' and 'evidence' columns with empty strings
        if "claim" in df.columns: df["claim"] = df["claim"].fillna("").astype(str)
        if "evidence" in df.columns: df["evidence"] = df["evidence"].fillna("").astype(str)

        # 1.1 Tách từ
        logger.info("  [1.1] Word segmentation …")
        df["claim_seg"]    = preprocessor.batch_segment(df["claim"].tolist())
        ev_col             = "evidence" if "evidence" in df.columns else df.columns[1]
        df["evidence_seg"] = preprocessor.batch_segment(df[ev_col].tolist())

        # 1.2 Phân rã tuyên bố
        logger.info("  [1.2] Claim decomposition …")
        df["sub_claims"] = df["claim_seg"].apply(
            lambda c: json.dumps(decomposer.decompose(c), ensure_ascii=False)
        )

        # 1.3 Lọc evidence
        logger.info("  [1.3] Evidence retrieval …")
        df["evidence_filtered"] = retriever.batch_retrieve(
            df["claim_seg"].tolist(),
            df["evidence_seg"].tolist(),
            use_sbert=cfg["use_sbert"],
        )

        processed[split_name] = df

        # Lưu parquet (tiết kiệm bộ nhớ so với csv)
        save_path = out / f"{split_name}_processed.parquet"
        df.to_parquet(save_path, index=False)
        logger.info(f"  ✓ Đã lưu → {save_path}")

        gc.collect()

    # Thống kê
    logger.info("\n── Thống kê nhãn ────────────────────────────────────────")
    for split_name, df in processed.items():
        dist = df["label"].value_counts().sort_index()
        dist.index = [ID2LABEL[i] for i in dist.index]
        logger.info(f"  {split_name}: {dist.to_dict()}")

    return processed

In [12]:
# ══════════════════════════════════════════════════════════════════════════════
# GIAI ĐOẠN 2 – XÂY DỰNG MÔ HÌNH (FINE-TUNING)
# ══════════════════════════════════════════════════════════════════════════════

In [14]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# ── Dataset ───────────────────────────────────────────────────────────────────
class FactCheckDataset(Dataset):
    """
    Format input: [CLS] Claim [SEP] Evidence [SEP]
    Mỗi phần tử được tokenize on-the-fly để không giữ tensor trong RAM.
    """

    def __init__(self, claims: List[str], evidences: List[str],
                 labels: List[int], tokenizer, max_length: int):
        self.claims     = claims
        self.evidences  = evidences
        self.labels     = labels
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.claims)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.claims[idx],
            self.evidences[idx],
            max_length   = self.max_length,
            padding      = "max_length",
            truncation   = True,
            return_tensors = "pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            # token_type_ids không có ở PhoBERT (RoBERTa-based) → zero
            "token_type_ids": enc.get(
                "token_type_ids",
                torch.zeros(self.max_length, dtype=torch.long)
            ).squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long),
        }


# ── Early Stopping ────────────────────────────────────────────────────────────
class EarlyStopping:
    def __init__(self, patience: int = 3, min_delta: float = 1e-4):
        self.patience   = patience
        self.min_delta  = min_delta
        self.counter    = 0
        self.best_score = None

    def step(self, score: float) -> bool:
        """Trả về True nếu nên dừng."""
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter    = 0
        else:
            self.counter += 1
        return self.counter >= self.patience


# ── Trainer ───────────────────────────────────────────────────────────────────
class FactCheckTrainer:
    """
    Fine-tuning PhoBERT / XLM-RoBERTa cho bài toán 3-class fact verification.

    Tối ưu bộ nhớ:
      • DataLoader num_workers=0 (tránh copy bộ nhớ fork)
      • Xóa cache CUDA sau mỗi epoch
      • Lưu best_model_state trên CPU
      • Mixed precision (fp16) nếu có CUDA
    """

    def __init__(self, cfg: dict):
        self.cfg    = cfg
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.use_fp16 = (self.device.type == "cuda")
        logger.info(f"Device: {self.device}  |  FP16: {self.use_fp16}")

        logger.info(f"Tải tokenizer & model: {cfg['model_name']} …")
        self.tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])
        self.model = AutoModelForSequenceClassification.from_pretrained(
            cfg["model_name"],
            num_labels     = NUM_LABELS,
            output_attentions = True,     # cần cho Phase 3
            ignore_mismatched_sizes = True,
        ).to(self.device)

        self.history: Dict[str, list] = defaultdict(list)
        self.best_model_state = None
        self._scaler = torch.cuda.amp.GradScaler() if self.use_fp16 else None

    # ── Helpers ───────────────────────────────────────────────────────────────
    def _make_loader(self, df: pd.DataFrame, shuffle: bool) -> DataLoader:
        ds = FactCheckDataset(
            claims    = df["claim_seg"].tolist(),
            evidences = df["evidence_filtered"].tolist(),
            labels    = df["label"].tolist(),
            tokenizer = self.tokenizer,
            max_length = self.cfg["max_length"],
        )
        return DataLoader(
            ds,
            batch_size  = self.cfg["batch_size"],
            shuffle     = shuffle,
            pin_memory  = (self.device.type == "cuda"),
            num_workers = 0,   # 0 = không fork → tiết kiệm RAM
        )

    @staticmethod
    def _class_weights(df: pd.DataFrame) -> torch.Tensor:
        counts = df["label"].value_counts().sort_index()
        w = torch.tensor([1.0 / c for c in counts.values], dtype=torch.float)
        return w / w.sum()   # normalize

    # ── Train ─────────────────────────────────────────────────────────────────
    def train(self, train_df: pd.DataFrame, dev_df: pd.DataFrame) -> None:
        train_loader = self._make_loader(train_df, shuffle=True)
        dev_loader   = self._make_loader(dev_df,   shuffle=False)

        weights   = self._class_weights(train_df).to(self.device)
        criterion = nn.CrossEntropyLoss(weight=weights)

        # AdamW + linear warmup + weight decay
        optimizer = torch.optim.AdamW(
            self.model.parameters(),
            lr           = self.cfg["lr"],
            weight_decay = 0.01,
        )
        total_steps  = len(train_loader) * self.cfg["epochs"]
        scheduler    = get_linear_schedule_with_warmup(
            optimizer,
            num_warmup_steps   = int(0.1 * total_steps),
            num_training_steps = total_steps,
        )
        early_stop = EarlyStopping(patience=self.cfg["patience"])

        for epoch in range(1, self.cfg["epochs"] + 1):
            # ── Train epoch ──────────────────────────────────────────────────
            self.model.train()
            tr_loss, correct, total = 0.0, 0, 0

            pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{self.cfg['epochs']} [Train]")
            for batch in pbar:
                ids  = batch["input_ids"].to(self.device, non_blocking=True)
                mask = batch["attention_mask"].to(self.device, non_blocking=True)
                ttids = batch["token_type_ids"].to(self.device, non_blocking=True)
                lbls  = batch["labels"].to(self.device, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)   # tiết kiệm hơn zero_grad()

                if self.use_fp16:
                    with torch.cuda.amp.autocast():
                        out  = self.model(input_ids=ids, attention_mask=mask)
                        loss = criterion(out.logits, lbls)
                    self._scaler.scale(loss).backward()
                    self._scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self._scaler.step(optimizer)
                    self._scaler.update()
                else:
                    out  = self.model(input_ids=ids, attention_mask=mask)
                    loss = criterion(out.logits, lbls)
                    loss.backward()
                    nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    optimizer.step()

                scheduler.step()
                tr_loss += loss.item()
                correct += (out.logits.argmax(-1) == lbls).sum().item()
                total   += lbls.size(0)
                pbar.set_postfix(loss=f"{loss.item():.4f}")

            avg_loss  = tr_loss / len(train_loader)
            train_acc = correct / total

            # ── Dev evaluation ────────────────────────────────────────────────
            dev_f1 = self._eval_f1(dev_loader)

            self.history["train_loss"].append(avg_loss)
            self.history["train_acc"].append(train_acc)
            self.history["dev_f1"].append(dev_f1)

            logger.info(
                f"Epoch {epoch:02d}  |  loss={avg_loss:.4f}  "
                f"train_acc={train_acc:.4f}  dev_f1={dev_f1:.4f}"
            )

            # Lưu model tốt nhất trên CPU (tiết kiệm VRAM)
            if (early_stop.best_score is None) or (dev_f1 >= early_stop.best_score):
                self.best_model_state = {
                    k: v.detach().cpu().clone()
                    for k, v in self.model.state_dict().items()
                }

            if early_stop.step(dev_f1):
                logger.info(f"⏹  Early stopping tại epoch {epoch}.")
                break

            # Dọn VRAM
            if self.device.type == "cuda":
                torch.cuda.empty_cache()
            gc.collect()

        # Khôi phục trọng số tốt nhất
        if self.best_model_state:
            self.model.load_state_dict(self.best_model_state)
            logger.info("✓ Đã khôi phục model tốt nhất.")

    # ── Eval F1 ───────────────────────────────────────────────────────────────
    @torch.no_grad()
    def _eval_f1(self, loader: DataLoader) -> float:
        from sklearn.metrics import f1_score
        self.model.eval()
        preds, golds = [], []
        for batch in loader:
            ids  = batch["input_ids"].to(self.device)
            mask = batch["attention_mask"].to(self.device)
            out  = self.model(input_ids=ids, attention_mask=mask)
            preds.extend(out.logits.argmax(-1).cpu().numpy())
            golds.extend(batch["labels"].numpy())
        return f1_score(golds, preds, average="macro", zero_division=0)

    # ── Predict ───────────────────────────────────────────────────────────────
    @torch.no_grad()
    def predict(self, df: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
        loader = self._make_loader(df, shuffle=False)
        self.model.eval()
        all_preds, all_probs = [], []
        for batch in tqdm(loader, desc="  Predicting"):
            ids  = batch["input_ids"].to(self.device)
            mask = batch["attention_mask"].to(self.device)
            out  = self.model(input_ids=ids, attention_mask=mask)
            probs = torch.softmax(out.logits, -1).cpu().numpy()
            all_probs.extend(probs)
            all_preds.extend(probs.argmax(-1))
        return np.array(all_preds), np.array(all_probs)

    # ── Save / Load ───────────────────────────────────────────────────────────
    def save(self, path: str) -> None:
        self.model.save_pretrained(path)
        self.tokenizer.save_pretrained(path)
        # Save training history
        history_path = Path(path).parent / "history.json"
        with open(history_path, "w", encoding="utf-8") as f:
            json.dump(self.history, f, ensure_ascii=False, indent=2)
        logger.info(f"Model và history đã lưu → {path} và {history_path}")

    def load(self, path: str) -> None:
        self.model = AutoModelForSequenceClassification.from_pretrained(
            path, output_attentions=True
        ).to(self.device)
        self.tokenizer = AutoTokenizer.from_pretrained(path)
        # Load training history
        history_path = Path(path).parent / "history.json"
        if history_path.exists():
            with open(history_path, "r", encoding="utf-8") as f:
                self.history = defaultdict(list, json.load(f))
            logger.info(f"Đã tải training history từ {history_path}")
        else:
            logger.warning(f"Không tìm thấy history.json tại {history_path}")


# ── Hàm Phase 2 chính ─────────────────────────────────────────────────────────
def run_phase2(processed: Dict[str, pd.DataFrame],
               cfg: dict) -> "FactCheckTrainer":
    logger.info("\n" + "═" * 70)
    logger.info("GIAI ĐOẠN 2: FINE-TUNING MÔ HÌNH")
    logger.info("═" * 70)

    splits = list(processed.keys())

    # Chọn split một cách linh hoạt
    train_df = processed.get("train", processed.get("train_set",
               processed[splits[0]]))
    dev_df   = processed.get("validation", processed.get("dev",
               processed.get("test",
               train_df.sample(frac=0.1, random_state=cfg["seed"]))));
    test_df  = processed.get("test", dev_df)

    logger.info(f"Train: {len(train_df)}  |  Dev: {len(dev_df)}  |  Test: {len(test_df)}")

    trainer = FactCheckTrainer(cfg)
    trainer.train(train_df, dev_df)
    trainer.save(str(Path(cfg["output_dir"]) / "model"))

    return trainer

In [15]:
# ══════════════════════════════════════════════════════════════════════════════
# GIAI ĐOẠN 3 – PHÂN TÍCH & TRỰC QUAN HÓA ATTENTION
# ══════════════════════════════════════════════════════════════════════════════

In [16]:
class AttentionAnalyzer:
    """
    Trích xuất và phân tích attention weights từ model đã fine-tune.

    Tính năng:
      • get_attention()         – lấy attention tất cả layers
      • plot_heatmap()          – heatmap token-level (không cần Jupyter)
      • bertviz_headview()      – BertViz head-view (cần Jupyter/HTML output)
      • numerical_check()       – kiểm tra attention vào token số
      • cross_attention_score() – điểm tập trung Claim→Evidence
    """

    def __init__(self, trainer: "FactCheckTrainer"):
        self.trainer   = trainer
        self.model     = trainer.model
        self.tokenizer = trainer.tokenizer
        self.device    = trainer.device

    # ── Lấy attention cho 1 mẫu ──────────────────────────────────────────────
    @torch.no_grad()
    def get_attention(self, claim: str, evidence: str) -> Dict:
        self.model.eval()
        enc = self.trainer.tokenizer(
            claim, evidence,
            max_length    = self.trainer.cfg["max_length"],
            padding       = "max_length",
            truncation    = True,
            return_tensors = "pt",
        ).to(self.device)

        out = self.model(**enc, output_attentions=True)

        tokens = self.tokenizer.convert_ids_to_tokens(
            enc["input_ids"][0].cpu().tolist()
        )
        # Cắt padding
        pad_tok = self.tokenizer.pad_token or "[PAD]"
        try:
            pad_idx = tokens.index(pad_tok)
        except ValueError:
            pad_idx = len(tokens)

        attns = [a.detach().cpu()[:, :, :pad_idx, :pad_idx]
                 for a in out.attentions]  # (1, H, T, T)
        tokens = tokens[:pad_idx]

        pred   = out.logits.argmax(-1).item()
        return {
            "tokens":     tokens,
            "attentions": attns,   # list[tensor] – 1 per layer
            "logits":     out.logits.cpu(),
            "pred":       pred,
            "pred_label": ID2LABEL[pred],
        }

    # ── Heatmap (Matplotlib) ──────────────────────────────────────────────────
    def plot_heatmap(self, claim: str, evidence: str,
                     layer: int = -1, head: int = 0,
                     max_tokens: int = 60, save_path: str = None) -> plt.Figure:
        data   = self.get_attention(claim, evidence)
        tokens = data["tokens"]
        attn   = data["attentions"][layer][0, head].numpy()   # (T, T)

        # Cắt để hiển thị
        n = min(max_tokens, len(tokens))
        tokens, attn = tokens[:n], attn[:n, :n]

        fig, ax = plt.subplots(figsize=(max(10, n // 2), max(8, n // 2)))
        sns.heatmap(
            attn, xticklabels=tokens, yticklabels=tokens,
            cmap="YlOrRd", ax=ax, linewidths=0.05, linecolor="grey",
        )
        ax.set_title(
            f"Attention  Layer {layer}  Head {head}\n"
            f"Dự đoán: {data['pred_label']}",
            fontsize=11,
        )
        plt.xticks(rotation=45, ha="right", fontsize=7)
        plt.yticks(rotation=0, fontsize=7)
        plt.tight_layout()

        if save_path:
            fig.savefig(save_path, dpi=150, bbox_inches="tight")
            logger.info(f"  Saved heatmap → {save_path}")
        return fig

    # ── BertViz (Jupyter / HTML) ──────────────────────────────────────────────
    def bertviz_headview(self, claim: str, evidence: str):
        """
        Trực quan hóa bằng BertViz.
        Chạy trong Jupyter Notebook để thấy widget.
        Nếu chạy ngoài Jupyter, trả về HTML string.
        """
        try:
            from bertviz import head_view
        except ImportError:
            logger.warning("bertviz chưa cài. Chạy: pip install bertviz")
            return None

        data  = self.get_attention(claim, evidence)
        attns = data["attentions"]   # list của (1, H, T, T)
        toks  = data["tokens"]

        # head_view cần list of (1, H, T, T)
        return head_view(attns, toks)

    # ── Cross-attention Claim→Evidence ───────────────────────────────────────
    def cross_attention_score(self, claim: str, evidence: str) -> Dict:
        """
        Tính điểm trung bình attention từ tokens của Claim sang tokens Evidence.
        Giúp xác định mô hình có "nhìn đúng" vào bằng chứng không.
        """
        data = self.get_attention(claim, evidence)
        toks = data["tokens"]

        # Tìm vị trí SEP để phân chia Claim và Evidence
        sep = self.tokenizer.sep_token or "</s>"
        sep_positions = [i for i, t in enumerate(toks) if t == sep]

        if len(sep_positions) < 2:
            logger.warning("Không tìm thấy đủ 2 SEP token.")
            return {}

        claim_end  = sep_positions[0]
        ev_start   = sep_positions[0] + 1
        ev_end     = sep_positions[1]

        claim_ids = list(range(1, claim_end))         # bỏ CLS
        ev_ids    = list(range(ev_start, ev_end))     # bỏ SEP cuối

        if not claim_ids or not ev_ids:
            return {}

        # Trung bình qua tất cả layers và heads
        all_attn = torch.stack(data["attentions"])    # (L, 1, H, T, T)
        avg_attn = all_attn.mean(dim=[0, 1, 2])       # (T, T)

        # Attention từ claim sang evidence (từng token claim trỏ sang ev)
        cross = avg_attn[np.ix_(claim_ids, ev_ids)].numpy()
        ev_toks = [toks[i] for i in ev_ids]

        # Top evidence tokens
        ev_scores = cross.mean(axis=0)               # (|ev|,)
        top_idx   = np.argsort(ev_scores)[-10:][::-1]

        return {
            "pred_label":    data["pred_label"],
            "cross_attn":    cross,
            "ev_tokens":     ev_toks,
            "top_ev_tokens": [(ev_toks[i], float(ev_scores[i])) for i in top_idx],
        }

    # ── Kiểm tra Numerical Reasoning ──────────────────────────────────────────
    def numerical_check(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Với mỗi mẫu chứa số liệu:
          • Tính trung bình attention vào các token numeric.
          • So sánh giữa mẫu đúng và mẫu sai.
        """
        NUM_PAT = re.compile(
            r"\d+|_PERCENT|_DONG|_USD|_TY|PERCENT|DONG|USD|TY|GOLD"
        )
        num_df = df[
            df["claim"].str.contains(r"\d", na=False)
            | df["claim_seg"].str.contains(r"_PERCENT|_DONG|_USD|_TY", na=False)
        ].copy()

        logger.info(f"  {len(num_df)} mẫu chứa số liệu → kiểm tra numerical attention …")

        rows = []
        for _, row in tqdm(num_df.iterrows(), total=len(num_df)):
            data  = self.get_attention(row["claim_seg"], row["evidence_filtered"])
            toks  = data["tokens"]
            num_i = [i for i, t in enumerate(toks) if NUM_PAT.search(t)]

            if not num_i:
                continue

            all_attn = torch.stack(data["attentions"]).mean(dim=[0, 1, 2])  # (T,T)
            # Trung bình attention được nhận bởi numeric tokens
            num_attn_recv = all_attn[:, num_i].mean().item()

            rows.append({
                "claim":          row["claim"],
                "true_label":     ID2LABEL[int(row["label"])],
                "pred_label":     data["pred_label"],
                "correct":        int(row["label"]) == data["pred"],
                "numeric_tokens": [toks[i] for i in num_i],
                "num_attn_recv":  round(num_attn_recv, 5),
            })

        result = pd.DataFrame(rows)
        logger.info(
            f"  Numerical accuracy: "
            f"{result['correct'].mean():.3f}  |  "
            f"Avg numeric attention: {result['num_attn_recv'].mean():.5f}"
        )
        return result


# ── Hàm Phase 3 chính ─────────────────────────────────────────────────────────
def run_phase3(trainer: "FactCheckTrainer",
               test_df: pd.DataFrame,
               cfg: dict) -> AttentionAnalyzer:
    logger.info("\n" + "═" * 70)
    logger.info("GIAI ĐOẠN 3: PHÂN TÍCH & TRỰC QUAN HÓA ATTENTION")
    logger.info("═" * 70)

    out      = Path(cfg["output_dir"])
    analyzer = AttentionAnalyzer(trainer)

    # Vẽ heatmap mỗi nhãn 1 mẫu
    for label_id, label_name in ID2LABEL.items():
        sub = test_df[test_df["label"] == label_id]
        if sub.empty:
            continue
        sample = sub.iloc[0]
        logger.info(f"\n▶ Vẽ attention heatmap mẫu nhãn [{label_name}] …")
        fig = analyzer.plot_heatmap(
            sample["claim_seg"],
            sample["evidence_filtered"],
            layer=-1, head=0,
            save_path=str(out / f"attention_heatmap_{label_name}.png"),
        )
        plt.close(fig)

    # Cross-attention minh họa cho 3 mẫu
    for i in range(min(3, len(test_df))):
        row    = test_df.iloc[i]
        ca     = analyzer.cross_attention_score(
            row["claim_seg"], row["evidence_filtered"]
        )
        if ca:
            logger.info(
                f"\n▶ Cross-attention top tokens (mẫu {i}):\n"
                f"  Pred={ca['pred_label']}  |  "
                f"Top evidence: {ca['top_ev_tokens'][:5]}"
            )

    # Numerical reasoning check
    num_result = analyzer.numerical_check(test_df)
    num_result.to_csv(out / "numerical_check.csv", index=False)
    logger.info(f"\n✓ Lưu kết quả numerical check → {out/'numerical_check.csv'}")

    return analyzer


In [17]:
# ══════════════════════════════════════════════════════════════════════════════
# GIAI ĐOẠN 4 – ĐÁNH GIÁ & BÁO CÁO (ERROR ANALYSIS)
# ══════════════════════════════════════════════════════════════════════════════

In [18]:
class Evaluator:
    """
    Đánh giá toàn diện:
      • Macro F1-Score theo từng nhãn.
      • Confusion Matrix.
      • Qualitative Error Analysis có giải thích Attention.
    """

    def __init__(self, trainer: "FactCheckTrainer",
                 analyzer: "AttentionAnalyzer"):
        self.trainer  = trainer
        self.analyzer = analyzer

    # ── Full evaluation ───────────────────────────────────────────────────────
    def evaluate(self, test_df: pd.DataFrame) -> Dict:
        from sklearn.metrics import classification_report, f1_score

        logger.info("  Đang predict trên tập test …")
        preds, probs = self.trainer.predict(test_df)
        labels       = test_df["label"].values

        macro_f1 = f1_score(labels, preds, average="macro", zero_division=0)
        report   = classification_report(
            labels, preds,
            target_names=[ID2LABEL[i] for i in range(NUM_LABELS)],
            zero_division=0,
        )

        logger.info(f"\n{'─'*60}")
        logger.info(f"MACRO F1 SCORE: {macro_f1:.4f}")
        logger.info(f"\n{report}")

        return {"preds": preds, "probs": probs, "labels": labels,
                "macro_f1": macro_f1, "report": report}

    # ── Confusion Matrix ──────────────────────────────────────────────────────
    def plot_confusion_matrix(self, labels: np.ndarray, preds: np.ndarray,
                              save_path: str = None) -> plt.Figure:
        from sklearn.metrics import confusion_matrix
        cm   = confusion_matrix(labels, preds)
        names = [ID2LABEL[i] for i in range(NUM_LABELS)]

        fig, ax = plt.subplots(figsize=(8, 6))
        sns.heatmap(
            cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=names, yticklabels=names, ax=ax,
            annot_kws={"size": 14},
        )
        ax.set_xlabel("Dự đoán", fontsize=12)
        ax.set_ylabel("Thực tế",  fontsize=12)
        ax.set_title("Confusion Matrix – ViFactCheck", fontsize=14)
        plt.tight_layout()

        if save_path:
            fig.savefig(save_path, dpi=150)
            logger.info(f"  Saved confusion matrix → {save_path}")
        return fig

    # ── Training curve ────────────────────────────────────────────────────────
    def plot_training_history(self, save_path: str = None) -> plt.Figure:
        h   = self.trainer.history
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        axes[0].plot(h["train_loss"], marker="o", color="#e74c3c", label="Train Loss")
        axes[0].set_title("Training Loss",  fontsize=12)
        axes[0].set_xlabel("Epoch"); axes[0].grid(alpha=0.3)

        axes[1].plot(h["dev_f1"], marker="s", color="#2ecc71", label="Dev Macro F1")
        axes[1].set_title("Dev Macro F1 Score", fontsize=12)
        axes[1].set_xlabel("Epoch"); axes[1].grid(alpha=0.3)

        plt.suptitle("ViFactCheck Fine-tuning Progress", fontsize=14, fontweight="bold")
        plt.tight_layout()

        if save_path:
            fig.savefig(save_path, dpi=150)
            logger.info(f"  Saved training curve → {save_path}")
        return fig

    # ── Error Analysis ────────────────────────────────────────────────────────
    def error_analysis(self, test_df: pd.DataFrame, preds: np.ndarray,
                       n_samples: int = 10) -> pd.DataFrame:
        df = test_df.copy()
        df["pred"]   = preds
        df["correct"] = df["pred"] == df["label"]

        errors = df[~df["correct"]]
        logger.info(f"\n  Tổng lỗi: {len(errors)}/{len(df)}")
        logger.info(
            f"  Phân phối lỗi (true→pred):\n"
            + errors.apply(
                lambda r: f"    {ID2LABEL[int(r['label'])]} → {ID2LABEL[int(r['pred'])]}",
                axis=1,
            ).value_counts().to_string()
        )

        samples = errors.sample(min(n_samples, len(errors)), random_state=42)
        records = []

        for _, row in tqdm(samples.iterrows(), total=len(samples),
                           desc="  Error analysis"):
            data  = self.analyzer.get_attention(
                row["claim_seg"], row["evidence_filtered"]
            )
            toks  = data["tokens"]
            last_attn = data["attentions"][-1][0].mean(0)  # avg heads (T,T)
            cls_row   = last_attn[0]                        # CLS → all tokens

            top5_i = cls_row.argsort()[-5:].tolist()
            top5   = [(toks[i], round(cls_row[i].item(), 4)) for i in top5_i]

            records.append({
                "claim":             row["claim"],
                "evidence_snippet":  str(row.get("evidence", ""))[:200],
                "true_label":        ID2LABEL[int(row["label"])],
                "pred_label":        ID2LABEL[int(row["pred"])],
                "top_attended":      top5,
                "explanation":       self._explain(
                    row, top5, ID2LABEL[int(row["label"])]
                ),
            })

        result = pd.DataFrame(records)
        return result

    @staticmethod
    def _explain(row: pd.Series, top_tokens: List[Tuple], true_label: str) -> str:
        NEG = {"không", "chưa", "chẳng", "chưa_hề", "không_hề", "không_phải"}
        NUM_PAT = re.compile(r"\d+|_PERCENT|_DONG|_USD|_TY|PERCENT")

        tok_names = [t[0] for t in top_tokens]
        has_neg   = any(t.lower() in NEG for t in tok_names)
        num_toks  = [t for t in tok_names if NUM_PAT.search(t)]

        if true_label == "REFUTED" and not has_neg:
            return (
                f"Mô hình bỏ qua từ phủ định trong tuyên bố. "
                f"Attention tập trung vào: {tok_names[-1]}."
            )
        if num_toks:
            return (
                f"Nhầm lẫn về số liệu {num_toks}. "
                f"Cần xem xét đơn vị và ngữ cảnh so sánh."
            )
        return (
            f"Mô hình chú trọng vào '{tok_names[-1]}' mà bỏ qua "
            f"ngữ cảnh quan trọng khác trong bằng chứng."
        )


In [19]:

# ══════════════════════════════════════════════════════════════════════════════
# MỞ RỘNG – XÂY DỰNG ĐỒ THỊ THỰC THỂ (GAT)
# ══════════════════════════════════════════════════════════════════════════════

In [20]:
class EntityGraphBuilder:
    """
    Xây dựng đồ thị đồng xuất hiện thực thể từ tập dữ liệu.
    Phục vụ Graph Attention Network (GAT) hoặc phân tích thông tin.
    """

    def __init__(self):
        try:
            from underthesea import ner as _ner
            self._ner_fn    = _ner
            self._has_ner   = True
        except Exception:
            self._has_ner = False
            logger.warning("NER của underthesea không khả dụng – dùng regex fallback.")

    def extract_entities(self, text: str) -> List[str]:
        if self._has_ner:
            try:
                tags = self._ner_fn(text)
                return [tok for tok, *_, tag in tags if tag != "O"]
            except Exception:
                pass
        # Fallback: từ viết hoa liên tiếp (tiếng Việt)
        return re.findall(
            r"[A-ZĐÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĂĐƠƯ][a-zđàáâãèéêìíòóôõùúýăđơư]+"
            r"(?:\s+[A-ZĐÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĂĐƠƯ][a-zđàáâãèéêìíòóôõùúýăđơư]+)*",
            text,
        )

    def build_graph(self, df: pd.DataFrame):
        import networkx as nx

        G = nx.Graph()
        ev_col = "evidence" if "evidence" in df.columns else df.columns[1]

        for _, row in tqdm(df.iterrows(), total=len(df), desc="  Build entity graph"):
            ents = list(set(
                self.extract_entities(str(row.get("claim", "")))
                + self.extract_entities(str(row.get(ev_col, "")))
            ))
            for e in ents:
                G.add_node(e)
                G.nodes[e]["count"] = G.nodes[e].get("count", 0) + 1

            for i in range(len(ents)):
                for j in range(i + 1, len(ents)):
                    if G.has_edge(ents[i], ents[j]):
                        G[ents[i]][ents[j]]["weight"] += 1
                    else:
                        G.add_edge(ents[i], ents[j], weight=1)

        logger.info(
            f"  Đồ thị: {G.number_of_nodes()} nút, {G.number_of_edges()} cạnh"
        )
        return G

    def visualize(self, G, top_n: int = 40, save_path: str = None) -> plt.Figure:
        import networkx as nx

        # Lấy top N nút theo bậc
        top_nodes = sorted(G.degree, key=lambda x: x[1], reverse=True)[:top_n]
        sub_G     = G.subgraph([n for n, _ in top_nodes]).copy()

        weights = [sub_G[u][v].get("weight", 1) for u, v in sub_G.edges()]
        degrees = dict(sub_G.degree())

        fig, ax = plt.subplots(figsize=(16, 12))
        pos     = nx.spring_layout(sub_G, k=1.8, seed=42)

        nx.draw_networkx_nodes(
            sub_G, pos,
            node_size  = [degrees[n] * 80 for n in sub_G.nodes()],
            node_color = "#3498db", alpha=0.8, ax=ax,
        )
        nx.draw_networkx_edges(
            sub_G, pos,
            width=[min(w / 2, 4) for w in weights],
            alpha=0.4, edge_color="#7f8c8d", ax=ax,
        )
        nx.draw_networkx_labels(sub_G, pos, font_size=7, ax=ax)

        ax.set_title(
            f"Đồ thị đồng xuất hiện thực thể – Top {top_n} nút",
            fontsize=14, fontweight="bold",
        )
        ax.axis("off")
        plt.tight_layout()

        if save_path:
            fig.savefig(save_path, dpi=150)
            logger.info(f"  Saved entity graph → {save_path}")
        return fig



In [21]:
# ── Hàm Phase 4 chính ─────────────────────────────────────────────────────────
def run_phase4(trainer: "FactCheckTrainer",
               analyzer: "AttentionAnalyzer",
               test_df: pd.DataFrame,
               all_df: pd.DataFrame,
               cfg: dict) -> None:
    logger.info("\n" + "═" * 70)
    logger.info("GIAI ĐOẠN 4: ĐÁNH GIÁ & BÁO CÁO (ERROR ANALYSIS)")
    logger.info("═" * 70)

    out       = Path(cfg["output_dir"])
    evaluator = Evaluator(trainer, analyzer)

    # 4.1 Đánh giá
    results = evaluator.evaluate(test_df)

    # 4.2 Confusion Matrix
    fig = evaluator.plot_confusion_matrix(
        results["labels"], results["preds"],
        save_path=str(out / "confusion_matrix.png"),
    )
    plt.close(fig)

    # 4.3 Training curve
    fig = evaluator.plot_training_history(
        save_path=str(out / "training_history.png"),
    )
    plt.close(fig)

    # 4.4 Lưu kết quả đánh giá
    report_path = out / "evaluation_report.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write(f"Model: {cfg['model_name']}\n")
        f.write(f"Macro F1: {results['macro_f1']:.4f}\n\n")
        f.write(results["report"])
    logger.info(f"\n  ✓ Saved evaluation report → {report_path}")

    # 4.5 Error analysis
    logger.info("\n▶ Phân tích lỗi …")
    error_df = evaluator.error_analysis(
        test_df, results["preds"], n_samples=10
    )
    error_df.to_csv(out / "error_analysis.csv", index=False, encoding="utf-8-sig")
    logger.info(f"  ✓ Saved error analysis → {out / 'error_analysis.csv'}")

    # In một vài lỗi tiêu biểu
    logger.info("\n── Mẫu lỗi tiêu biểu ──────────────────────────────────────")
    for _, row in error_df.head(3).iterrows():
        logger.info(
            f"\n  Claim:     {row['claim'][:100]}\n"
            f"  Thực tế:   {row['true_label']}  →  Dự đoán: {row['pred_label']}\n"
            f"  Top attn:  {row['top_attended']}\n"
            f"  Giải thích: {row['explanation']}"
        )

    # 4.6 Entity graph (GAT prep)
    logger.info("\n▶ Xây dựng đồ thị thực thể …")
    graph_builder = EntityGraphBuilder()
    G   = graph_builder.build_graph(all_df)
    fig = graph_builder.visualize(G, save_path=str(out / "entity_graph.png"))
    plt.close(fig)

    logger.info("\n✅ Giai đoạn 4 hoàn tất.")



In [23]:
import random
import numpy as np
import torch

def seed_everything(seed: int):
    """Đặt seed cho tất cả các thư viện để tăng khả năng tái tạo kết quả."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # cho nhiều GPU
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # os.environ['PYTHONHASHSEED'] = str(seed) # Có thể ảnh hưởng đến hiệu suất và không khuyến khích trong môi trường sản xuất
    logger.info(f"Đã đặt seed cho tất cả các thư viện thành {seed}.")


In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# PIPELINE CHÍNH
# ══════════════════════════════════════════════════════════════════════════════

def run_pipeline(phase: Optional[int] = None, **override) -> None:
    """
    Chạy toàn bộ pipeline hoặc 1 giai đoạn cụ thể.

    Args:
        phase: None = toàn bộ; 1/2/3/4 = giai đoạn cụ thể.
        **override: Override bất kỳ key nào trong CFG.
    """
    cfg = {**CFG, **override}
    Path(cfg["output_dir"]).mkdir(parents=True, exist_ok=True)

    # --- Gọi seed_everything ngay từ đầu ---
    seed_everything(cfg["seed"])
    # ---------------------------------------

    logger.info("╔" + "═" * 68 + "╗")
    logger.info(f"║  ViFactCheck Pipeline  –  Model: {cfg['model_name']:<30}  ║")
    logger.info("╚" + "═" * 68 + "╝")
    logger.info(f"Cấu hình: {json.dumps(cfg, indent=2, ensure_ascii=False)}")

    out = Path(cfg["output_dir"])

    # ── Phase 1 ──────────────────────────────────────────────────────────────
    processed = None
    if phase is None or phase == 1:
        raw_splits = load_data()
        if "sample_size" in cfg and cfg["sample_size"] > 0:
            sampled_splits = {}
            for name, df in raw_splits.items():
                if len(df) > cfg["sample_size"]:
                    sampled_splits[name] = df.sample(cfg["sample_size"], random_state=cfg["seed"])
                    logger.info(f"  Đã lấy mẫu {cfg['sample_size']} từ split '{name}'.")
                else:
                    sampled_splits[name] = df
            processed = run_phase1(cfg, sampled_splits)
        else:
            processed = run_phase1(cfg, raw_splits)
    elif phase in (2, 3, 4):
        # Load từ parquet đã lưu
        parquet_files = list(out.glob("*_processed.parquet"))
        if not parquet_files:
            raise FileNotFoundError(
                "Chưa có dữ liệu đã xử lý. Chạy Phase 1 trước."
            )
        processed = {f.stem.replace("_processed", ""): pd.read_parquet(f)
                     for f in parquet_files}
        logger.info(f"Đã tải {len(processed)} splits từ {out}")

    # Chọn splits
    split_names = list(processed.keys())
    train_df = processed.get("train", processed.get("train_set",
               processed[split_names[0]]))
    dev_df   = processed.get("validation", processed.get("dev",
               processed.get("test",
               train_df.sample(frac=0.1, random_state=cfg["seed"]))))
    test_df  = processed.get("test", dev_df)
    all_df   = pd.concat(list(processed.values()), ignore_index=True)

    # ── Phase 2 ──────────────────────────────────────────────────────────────
    trainer = None
    if phase is None or phase == 2:
        trainer = run_phase2(processed, cfg)
    elif phase in (3, 4):
        model_path = str(out / "model")
        if not Path(model_path).exists():
            raise FileNotFoundError(f"Model không tồn tại tại {model_path}. Chạy Phase 2 trước.")
        trainer = FactCheckTrainer(cfg)
        trainer.load(model_path)

    if trainer is None:
        return   # Phase 1 only

    # ── Phase 3 ──────────────────────────────────────────────────────────────
    analyzer = None
    if phase is None or phase == 3:
        analyzer = run_phase3(trainer, test_df, cfg)
    elif phase == 4:
        analyzer = AttentionAnalyzer(trainer)

    if analyzer is None:
        return   # Phase 1+2 only

    # ── Phase 4 ──────────────────────────────────────────────────────────────
    if phase is None or phase == 4:
        run_phase4(trainer, analyzer, test_df, all_df, cfg)

    logger.info(
        f"\n🎉 Pipeline hoàn tất! Kết quả tại: {out.resolve()}"
    )

In [25]:
# ══════════════════════════════════════════════════════════════════════════════
# HIỂN THỊ KẾT QUẢ – GỌI SAU KHI pipeline hoàn tất
# ═════════════════════════════════════════════════════════════════════════════

In [26]:
import matplotlib
import pandas as pd

def display_results(output_dir: str = "./vifactcheck_output") -> None:
    """
    In và hiển thị toàn bộ kết quả đã lưu:
      • Thống kê dữ liệu đã xử lý
      • Evaluation report (F1 từng nhãn)
      • Confusion Matrix (inline)
      • Training curve (inline)
      • Top-5 lỗi tiêu biểu (có giải thích Attention)
      • Numerical reasoning summary
      • Attention heatmap (inline)
    Chạy được trong cả Jupyter/Colab và terminal thuần.
    """
    import sys, textwrap
    from pathlib import Path
    import json

    out = Path(output_dir)

    # ── Detect môi trường ──────────────────────────────────────────────────────
    in_notebook = "ipykernel" in sys.modules
    if in_notebook:
        from IPython.display import display, Image as IPImage, HTML
        matplotlib.use("inline")          # Đảm bảo hiển thị trong notebook

    def _sep(title: str = "", char: str = "═", width: int = 70) -> None:
        if title:
            pad = (width - len(title) - 2) // 2
            print(char * pad + f" {title} " + char * (width - pad - len(title) - 2))
        else:
            print(char * width)

    def _show_image(path: Path, caption: str = "") -> None:
        if not path.exists():
            print(f"  [Chưa có file: {path.name}]")
            return
        if in_notebook:
            display(IPImage(filename=str(path)))
            if caption:
                display(HTML(f"<p style='text-align:center;color:gray'>{caption}</p>"))
        else:
            print(f"  → Xem ảnh tại: {path.resolve()}")

    print("\n")
    _sep("📊 TỔNG KẾT KẾT QUẢ VIFACTCHECK PIPELINE", "═")

      # ── 1. Thống kê dữ liệu đã xử lý ─────────────────────────────────────────
    _sep("1. DỮ LIỆU ĐÃ XỬ LÝ", "─")
    parquet_files = sorted(out.glob("*_processed.parquet"))
    if parquet_files:
        for pf in parquet_files:
            df = pd.read_parquet(pf)
            split_name = pf.stem.replace("_processed", "")
            label_dist = df["label"].value_counts().sort_index()
            label_str  = "  |  ".join(
                f"{ID2LABEL[int(k)]}: {v}" for k, v in label_dist.items()
            )
            print(f"\n  Split '{split_name}':  {len(df):,} mẫu")
            print(f"    Nhãn  →  {label_str}")

            # Mẫu claim + sub-claims
            sample = df.iloc[0]
            print(f"\n  Ví dụ claim gốc   : {str(sample.get('claim',''))[:120]}")
            print(f"  Sau word-segment  : {str(sample.get('claim_seg',''))[:120]}")
            sub = json.loads(sample.get("sub_claims", '[""]'))
            print(f"  Sub-claims        : {sub}")
            print(f"  Evidence filtered : {str(sample.get('evidence_filtered',''))[:200]}")
    else:
        print("  Chưa có dữ liệu – chạy Phase 1 trước.")

    # ── 2. Evaluation Report ───────────────────────────────────────────────────
    _sep("2. KẾT QUẢ ĐÁNH GIÁ (CLASSIFICATION REPORT)", "─")
    report_path = out / "evaluation_report.txt"
    if report_path.exists():
        with open(report_path, encoding="utf-8") as f:
            content = f.read()
        print(content)
    else:
        print("  Chưa có – chạy Phase 4 trước.")

    # ── 3. Confusion Matrix ────────────────────────────────────────────────────
    _sep("3. CONFUSION MATRIX", "─")
    _show_image(out / "confusion_matrix.png",
                "Confusion Matrix – SUPPORTED / REFUTED / NEI")

    # ── 4. Training Curve ──────────────────────────────────────────────────────
    _sep("4. TRAINING CURVE (Loss & Macro F1)", "─")
    _show_image(out / "training_history.png",
                "Train Loss  &  Dev Macro F1 theo Epoch")

    # ── 5. Error Analysis ──────────────────────────────────────────────────────
    _sep("5. PHÂN TÍCH LỖI TIÊU BIỂU (TOP 5)", "─")
    err_path = out / "error_analysis.csv"
    if err_path.exists():
        err_df = pd.read_csv(err_path)
        for i, row in err_df.head(5).iterrows():
            print(f"\n  ── Lỗi #{i+1} ──────────────────────────────────────────")
            print(f"  Claim     : {textwrap.fill(str(row['claim'])[:200], 70, subsequent_indent='              ')}")
            print(f"  Evidence  : {str(row['evidence_snippet'])[:180]} …")
            print(f"  Thực tế   : {row['true_label']}  →  Dự đoán: {row['pred_label']}")
            print(f"  Top Attn  : {row['top_attended']}")
            print(f"  Giải thích: {textwrap.fill(str(row['explanation']), 68, subsequent_indent='              ')}")
    else:
        print("  Chưa có – chạy Phase 4 trước.")

    # ── 6. Numerical Reasoning ────────────────────────────────────────────────
    _sep("6. NUMERICAL REASONING CHECK", "─")
    num_path = out / "numerical_check.csv"
    if num_path.exists():
        num_df = pd.read_csv(num_path)
        total   = len(num_df)
        correct = num_df["correct"].sum()
        acc     = correct / total if total else 0
        avg_attn = num_df["num_attn_recv"].mean()

        print(f"\n  Tổng mẫu có số liệu  : {total}")
        print(f"  Dự đoán đúng          : {correct}  ({acc:.1%})")
        print(f"  Avg attention → số    : {avg_attn:.5f}")

        # So sánh đúng / sai
        if "correct" in num_df.columns:
            g = num_df.groupby("correct")["num_attn_recv"].mean()
            for k, v in g.items():
                label = "Đúng ✓" if k else "Sai ✗"
                print(f"    {label}:  avg numeric attention = {v:.5f}")

        # Mẫu sai có số liệu
        wrong = num_df[~num_df["correct"]]
        if not wrong.empty:
            print(f"\n  Mẫu sai về số liệu (top 3):")
            for _, r in wrong.head(3).iterrows():
                print(f"    Claim: {str(r['claim'])[:100]}")
                print(f"    Nhãn thực: {r['true_label']}  |  Dự đoán: {r['pred_label']}")
                print(f"    Numeric tokens: {r['numeric_tokens']}")
                print()
    else:
        print("  Chưa có – chạy Phase 3 trước.")

    # ── 7. Attention Heatmap ──────────────────────────────────────────────────
    _sep("7. ATTENTION HEATMAP (mỗi nhãn 1 mẫu)", "─")
    for label_name in ID2LABEL.values():
        img_path = out / f"attention_heatmap_{label_name}.png"
        print(f"\n  Nhãn: {label_name}")
        _show_image(img_path, f"Attention heatmap – {label_name}")

    # ── 8. Entity Graph ───────────────────────────────────────────────────────
    _sep("8. ĐỒ THỊ THỰC THỂ (Entity Graph)", "─")
    _show_image(out / "entity_graph.png",
                "Đồ thị đồng xuất hiện thực thể kinh tế")

    # ── 9. So sánh trực quan nhãn: attention heatmap grid ─────────────────────
    _sep("9. SO SÁNH LABEL – ATTENTION GRID", "─")
    imgs = [out / f"attention_heatmap_{n}.png" for n in ID2LABEL.values()]
    valid = [p for p in imgs if p.exists()]
    if valid:
        fig, axes = plt.subplots(1, len(valid), figsize=(7 * len(valid), 6))
        if len(valid) == 1:
            axes = [axes]
        for ax, p in zip(axes, valid):
            img = plt.imread(str(p))
            ax.imshow(img)
            ax.set_title(p.stem.replace("attention_heatmap_", "Nhãn: "),
                        fontsize=13, fontweight="bold")
            ax.axis("off")
        plt.suptitle("So sánh Attention theo từng nhãn phán quyết",
                     fontsize=15, fontweight="bold", y=1.01)
        plt.tight_layout()
        grid_path = out / "attention_grid_comparison.png"
        plt.savefig(grid_path, dpi=120, bbox_inches="tight")
        if in_notebook:
            plt.show()
        else:
            print(f"  → Xem ảnh tại: {grid_path.resolve()}")
        plt.close()

    _sep("✅ HIỂN THỊ HOÀN TẤT", "═")
    print(f"\n  Toàn bộ kết quả được lưu tại: {out.resolve()}\n")

In [27]:
run_pipeline(phase=1)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/8.07M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5062 [00:00<?, ? examples/s]

...

  Evidence filter:   0%|          | 0/141 [00:00<?, ?it/s]

DEBUG (run_phase1): Columns in 'dev' split: ['unnamed: 0', 'index', 'claim', 'context', 'annotation_id', 'topic', 'author', 'url', 'label', 'evidence']


  Word-segment:   0%|          | 0/1 [00:00<?, ?it/s]

  Word-segment:   0%|          | 0/1 [00:00<?, ?it/s]

  Evidence filter:   0%|          | 0/67 [00:00<?, ?it/s]

In [28]:
run_pipeline(phase=2)

config.json:   0%|          | 0.00/557 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/543M [00:00<?, ?B/s]

...

Epoch 9/10 [Train]:   0%|          | 0/60 [00:00<?, ?it/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Epoch 10/10 [Train]:   0%|          | 0/60 [00:00<?, ?it/s]

Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [29]:
run_pipeline(phase=3)

In [ ]:
run_pipeline(phase=4)

In [ ]:
display_results()

In [ ]:
import pandas as pd
import json
from pathlib import Path

output_dir = "./vifactcheck_output"
out = Path(output_dir)

print("--- Nội dung file evaluation_report.txt ---")
report_path = out / "evaluation_report.txt"
if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print(f"File không tồn tại: {report_path}")

print("\n--- Nội dung file history.json (chỉ 5 dòng cuối) ---")
history_path = out / "history.json"
if history_path.exists():
    with open(history_path, "r", encoding="utf-8") as f:
        history_data = json.load(f)
        print(json.dumps(history_data, indent=2))
else:
    print(f"File không tồn tại: {history_path}")

print("\n--- Nội dung file numerical_check.csv (chỉ 5 dòng đầu) ---")
num_check_path = out / "numerical_check.csv"
if num_check_path.exists():
    num_df = pd.read_csv(num_check_path)
    print(num_df.head().to_markdown(index=False))
    print(f"\nTổng số mẫu có số liệu: {len(num_df)}")
    print(f"Số mẫu dự đoán đúng: {num_df['correct'].sum()} ({num_df['correct'].mean():.2%})")
    print(f"Avg attention vào số: {num_df['num_attn_recv'].mean():.5f}")
else:
    print(f"File không tồn tại: {num_check_path}")

print("\n--- Nội dung file error_analysis.csv (chỉ 3 dòng đầu) ---")
error_path = out / "error_analysis.csv"
if error_path.exists():
    error_df = pd.read_csv(error_path)
    print(error_df.head(3).to_markdown(index=False))
else:
    print(f"File không tồn tại: {error_path}")

print("\n--- Thống kê dữ liệu đã xử lý (từ các file parquet) ---")
parquet_files = list(out.glob("*_processed.parquet"))
if parquet_files:
    for pf in parquet_files:
        df_processed = pd.read_parquet(pf)
        split_name = pf.stem.replace("_processed", "")
        label_dist = df_processed["label"].value_counts().sort_index()
        label_str  = "  |  ".join(
            f"{ID2LABEL[int(k)]}: {v}" for k, v in label_dist.items()
        )
        print(f"  Split '{split_name}':  {len(df_processed):,} mẫu")
        print(f"    Nhãn  →  {label_str}")
        # In một ví dụ cụ thể cho dữ liệu đã xử lý
        if not df_processed.empty:
            sample = df_processed.iloc[0]
            print(f"  Ví dụ claim gốc   : {str(sample.get('claim',''))[:120]}")
            print(f"  Sau word-segment  : {str(sample.get('claim_seg',''))[:120]}")
            sub = json.loads(sample.get("sub_claims", '[""]'))
            print(f"  Sub-claims        : {sub}")
            print(f"  Evidence filtered : {str(sample.get('evidence_filtered',''))[:200]}")
        print("\n")
else:
    print("Chưa có dữ liệu xử lý trong thư mục output.")
